In [ ]:
import os
from glob import glob
from functools import partial
from natsort import natsorted
import json

from torch.utils.data import ConcatDataset, random_split
from torchvision.transforms import v2

from data import SparseLabeledImageDataset, NormalizationStrategy, ZeroChannelDropout


In [ ]:
dataset_paths = [
    {
        'base_path': '/data/agl_data/AndreasMaiser/NSD/26AM06-02_1',
        'image_subfolder': 'patches_gfp+',
        'label_subfolder': 'patches-segmentation-threshold',
        'image_file_pattern': "*_ch0*.tif",
        'label_file_pattern': "*.tif"
    },
    {
        'base_path': '/data/agl_data/AndreasMaiser/NSD/26AM06-02_2',
        'image_subfolder': 'patches_gfp+',
        'label_subfolder': 'patches-segmentation-threshold',
        'image_file_pattern': "*_ch1*.tif",
        'label_file_pattern': "*.tif"
    }
]

In [ ]:
# assemble image and label files lists from one or more dataset_paths
img_files = []
label_files = []
for dataset_path in dataset_paths:
    img_files_i = natsorted(glob(os.path.join(dataset_path['base_path'], dataset_path['image_subfolder'], dataset_path['image_file_pattern'])))
    label_files_i = natsorted(glob(os.path.join(dataset_path['base_path'], dataset_path['label_subfolder'], dataset_path['label_file_pattern'])))
    img_files.extend(img_files_i)
    label_files.extend(label_files_i)

# random resize crop (should work even for smaller img) and flips
tr = v2.Compose(
    [
        v2.RandomResizedCrop((128,128), scale=(0.9, 1.0)),
        v2.RandomHorizontalFlip(),
        v2.RandomVerticalFlip(),
        ZeroChannelDropout(keep_idx=2)
    ]
)

# plane selector funtions
# NOTE: we first select labelled planes, than the middle of those
selectors = [
    # partial(get_labeled_planes_selection, min_labeled_pixels=20),
    # partial(get_mid_planes_selection, q=0.5),
]

dataset_train = SparseLabeledImageDataset(img_files, label_files, transforms=tr,
                                    plane_selectors=selectors,
                                    normalization_strategy=NormalizationStrategy.PER_IMAGE,
                                    plane_sliding_window=5)
dataset_val = None

In [ ]:
from matplotlib import pyplot as plt
from random import randint

img, mask = dataset_train[randint(0, len(dataset_train))]

fig, axs = plt.subplots(ncols=2)
axs[0].imshow(img.max(axis=0)[0])
axs[1].imshow(mask.squeeze())

len(dataset_train)

In [ ]:
from math import ceil
from torch.utils.data import DataLoader
from lightning import pytorch as L

from segmentation_unet_train import DenseSegmentationUNet

net = DenseSegmentationUNet(1, [64, 128, 128], input_channels=5)
loader = DataLoader(dataset_train, batch_size=64, shuffle=True)

trainer = L.Trainer(
    logger=L.loggers.CSVLogger(""),
    log_every_n_steps=ceil(len(dataset_train) / loader.batch_size),
    max_epochs=300,
)

trainer.fit(net, loader)

In [ ]:
from lightning.pytorch.utilities.model_summary import ModelSummary

net_inference =  LightningUNet.load_from_checkpoint('lightning_logs/version_34/checkpoints/epoch=19-step=5660.ckpt').eval()
# net_inference =  LightningUNet.load_from_checkpoint('/Users/david/Desktop/jurkat_nucleolin/unet_nucleolus_001/checkpoints/epoch=299-step=3300.ckpt').eval()


ModelSummary(net_inference, max_depth=3)

In [ ]:
test_file = '/Volumes/nn/Julia Vogtmann/Microscopy/26JV_018/tif/0001_ch0.tif'

# add two dummy dimensions (batch size, channels)
img = torch.from_numpy(imread(test_file)).float()[:, torch.newaxis, torch.newaxis]

# ALTERNATIVE with full loader (different batch size, etc.):
# img = torch.from_numpy(imread(test_file)).float()[:, torch.newaxis]
# predict_ds = torch.utils.data.TensorDataset(img)
# predict_loader = torch.utils.data.DataLoader(predict_ds, 1)


trainer = L.Trainer(enable_checkpointing=False, logger=False)
with torch.no_grad():
    pred = trainer.predict(net_inference, img)
    pred = torch.concat(pred)
    probs = torch.softmax(pred, 1)
    pred_labels = pred.argmax(1)

In [ ]:
import nd2

test_file = '/Volumes/agl_data/AndreasMaiser/NSD/26AM06-02_2/0010.nd2'
img = nd2.imread(test_file, dask=True, xarray=True)
img = img.isel(C=1).values
img = SparseLabeledImageDataset._normalize_intensities(img, NormalizationStrategy.PER_IMAGE)

sliding_window = 5
if sliding_window > 1:
    img = sliding_window_planewise_padded(img, sliding_window)
    img = torch.from_numpy(img).float()[:, torch.newaxis, :]
else:
    img = torch.from_numpy(img).float()[:, torch.newaxis, torch.newaxis]


trainer = L.Trainer(enable_checkpointing=False, logger=False)
with torch.no_grad():
    pred = trainer.predict(net_inference, img)
    pred = torch.concat(pred)

    if pred.shape[1] == 1:
        probs = torch.sigmoid(pred[:,0])
        pred_labels = probs > 0.5
    else:
        probs = torch.softmax(pred, 1)
        pred_labels = pred.argmax(1)

In [ ]:
import napari

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.Viewer()
viewer.add_image(img[:,0,sliding_window//2])

# view predictions of a single class
# viewer.add_labels((pred_labels==2).int())

# ALTERNATIVE: all classes
viewer.add_labels(pred_labels.int())


viewer.add_image(probs)